In [1]:
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph.message import add_messages

c:\Users\ASHUTOSH\anaconda3\envs\tf_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ASHUTOSH\anaconda3\envs\tf_env\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
load_dotenv()
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('D:/Data Analysis/Space Mission Analysis/Dataset/Space_Missions_Dataset.csv')
df['Launch_Year'] = df['Launch_Date'].str.split("-").str[0]

In [4]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
)

In [5]:
from langchain_core.documents import Document

documents = []

for _, row in df.iterrows():
    content = "\n".join(
        f"{col}: {row[col]}"
        for col in df.columns
    )

    documents.append(
        Document(
            page_content=content,
            metadata = {
                "mission_name": row["Mission_Name"],
                "agency": row["Agency"],
                "launch_year": row["Launch_Year"]
            }
        )
    )

print(documents[0].page_content)

Mission_ID: NA-00001
Mission_Name: Explorer 1
Agency: NASA
Country_Region: USA
Agency_Type: Government
Program_Type: Technology Demo
Mission_Category: Mercury
Sub_Category: Orbiter
Launch_Date: 1976-01-12
End_Date: 1987-02-18
Duration: 11.1 years
Launch_Vehicle: Pegasus
Launch_Site: Kennedy Space Center
Status: Failed
Mission_Phase: Past
Crew_Type: Uncrewed
Crew_Members: nan
Destination: Mercury
Objective: Black hole imaging and stellar cataloguing
Key_Achievement: Debris impact damaged solar panels
Cost_USD_Million: 7708.6
Partner_Agencies: nan
Data_Returned: Partial
Failure_Reason: Debris impact damaged solar panels
Mission_Outcome_Detail: Exceeded planned science return by 3x
Reference_URL: https://www.nasa.gov/mission/explorer-1
Launch_Year: 1976


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# vector_store = FAISS.from_documents(documents, embeddings)
# vector_store.save_local("faiss_index")


In [7]:
vector_store = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [8]:
retriever = vector_store.as_retriever(search_type = 'similarity', search_kwargs={'k':4})

In [9]:
retriever.invoke('What is Apollo 11')

[Document(id='8623c0ad-5929-4d0d-a46c-8c2c53773b95', metadata={'mission_name': 'Apollo 11', 'agency': 'NASA', 'launch_year': '1961'}, page_content='Mission_ID: NA-00013\nMission_Name: Apollo 11\nAgency: NASA\nCountry_Region: USA\nAgency_Type: Government\nProgram_Type: Commercial\nMission_Category: Mercury\nSub_Category: Flyby\nLaunch_Date: 1961-05-25\nEnd_Date: 1969-09-12\nDuration: 8.3 years\nLaunch_Vehicle: Pegasus\nLaunch_Site: Kennedy Space Center\nStatus: Failed\nMission_Phase: Past\nCrew_Type: Uncrewed\nCrew_Members: nan\nDestination: Mercury\nObjective: Next-generation communication satellite deployment\nKey_Achievement: nan\nCost_USD_Million: 9085.2\nPartner_Agencies: nan\nData_Returned: No\nFailure_Reason: nan\nMission_Outcome_Detail: Failed to achieve orbit due to launch vehicle anomaly\nReference_URL: https://www.nasa.gov/mission/apollo-11\nLaunch_Year: 1961'),
 Document(id='430ad394-1681-4acc-9f00-1e782a55599d', metadata={'mission_name': 'Apollo 13', 'agency': 'NASA', 'laun

In [10]:
from typing import TypedDict, Annotated
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [11]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template = """
       You are AstroGuide, an expert AI assistant specializing in space missions.

You are provided with mission documents retrieved from a trusted space mission database.

Rules:

• Use the retrieved context as your primary source.
• Never invent mission facts.
• If the answer is partially available, clearly state what is known.
• If the retrieved context is insufficient, respond:
  "I couldn't find enough information in the mission database."

• When appropriate:
  - Explain concepts simply.
  - Use bullet points.
  - Mention mission objectives, agencies, launch year, destination, achievements and mission status.

• If multiple missions are retrieved, compare them logically.

Retrieved Context:
{context}

Question:
{question}

Provide a detailed yet concise answer.

At the end include:

Source Missions:
- <mission names from retrieved documents>
    """,
    input_variables = ['context', 'question']
)

In [12]:
question = "Which country launched Aryabhata"
retrieved_docs = retriever.invoke(question)

In [13]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'Mission_ID: IS-00618\nMission_Name: Aryabhata\nAgency: ISRO\nCountry_Region: India\nAgency_Type: Government\nProgram_Type: Telescope\nMission_Category: Earth Observation\nSub_Category: Climate Study\nLaunch_Date: 2013-09-16\nEnd_Date: 2023-03-20\nDuration: 9.5 years\nLaunch_Vehicle: SLV\nLaunch_Site: Thumba Equatorial Rocket Station\nStatus: Failed\nMission_Phase: Past\nCrew_Type: Uncrewed\nCrew_Members: nan\nDestination: Earth Observation\nObjective: Asteroid composition and deflection study\nKey_Achievement: Landing system failure on descent\nCost_USD_Million: 8032.8\nPartner_Agencies: nan\nData_Returned: No\nFailure_Reason: Landing system failure on descent\nMission_Outcome_Detail: First images returned from target destination\nReference_URL: https://www.isro.gov.in/mission/aryabhata\nLaunch_Year: 2013\n\nMission_ID: JA-00756\nMission_Name: Hayabusa 3\nAgency: JAXA\nCountry_Region: Japan\nAgency_Type: Government\nProgram_Type: Technology Demo\nMission_Category: Jupiter\nSub_Categor

In [14]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
response = llm.invoke(final_prompt)
print(response.content)

**Answer**

- **Country that launched Aryabhata:** **India**  
  - The mission was carried out by the Indian Space Research Organisation (ISRO), which is the governmental space agency of India.

**Source Missions**
- Aryabhata  
- Hayabusa 3  
- Hayabusa  
- Chandrayaan‑2
